# Regenerate and Persist Missing Embeddings
This notebook connects to Drive, verifies GPU, and regenerates missing ReID, DINOv3, and tracklet embeddings.

To run this in Colab, upload this file, ensure the runtime is set to GPU, and run all cells.

In [1]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

In [2]:
# ==============================================================================
# Step a: Mount Drive and set environment variable
# ==============================================================================
print("[STEP A] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])

[STEP A] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ==============================================================================
# Step b: Verify GPU
# ==============================================================================
print("\n[STEP B] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")


[STEP B] Verifying GPU...
[PASS] GPU detected: Tesla T4


In [4]:
# ==============================================================================
# Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

def verify_file(path_str):
    p = Path(path_str)
    if p.exists() and p.stat().st_size > 0:
        return True, p.stat().st_size
    return False, 0

M	data_manifests/reid_summary.csv
Already on 'feature/stage1-object-detection'
Your branch is up to date with 'origin/feature/stage1-object-detection'.
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
 * branch            feature/stage1-object-detection -> FETCH_HEAD
Already up to date.


In [11]:
# ==============================================================================
# Step c: Run ReID Feature Extraction
# ==============================================================================
print("\n" + "="*50)
print("[STEP C] Running ReID Feature Extraction...")
print("="*50)
!python -m src.extract_reid_features

print("\nVerifying ReID Outputs...")
reid_files = [
    DRIVE_ROOT / "data" / "embeddings" / "view1_embeddings.npz",
    DRIVE_ROOT / "data" / "embeddings" / "view2_embeddings.npz",
    DRIVE_ROOT / "data" / "embeddings" / "view3_embeddings.npz",
]

reid_passed = True
reid_results = {}
for f in reid_files:
    exists, size = verify_file(f)
    if exists:
        print(f"[PASS] {f.name} exists with size {size} bytes.")
        reid_results[f.name] = size
    else:
        print(f"[FAIL] {f.name} is missing or empty.")
        reid_passed = False

if not reid_passed:
    print("[FAIL] Step C failed verification. Halting.")
    sys.exit(1)


[STEP C] Running ReID Feature Extraction...
[INFO] Initializing ReID model (osnet_x1_0) on cuda
[INFO] Loaded ReID weights from /content/drive/MyDrive/CCTV-Multiview-Project/checkpoints/osnet_x1_0_market1501.pth (565/565 keys matched)
[SKIP] view1: ReID embeddings already present at /content/drive/MyDrive/CCTV-Multiview-Project/data/embeddings/view1_embeddings.npz (use --force to re-run)
[SKIP] view2: ReID embeddings already present at /content/drive/MyDrive/CCTV-Multiview-Project/data/embeddings/view2_embeddings.npz (use --force to re-run)
[SKIP] view3: ReID embeddings already present at /content/drive/MyDrive/CCTV-Multiview-Project/data/embeddings/view3_embeddings.npz (use --force to re-run)
[DONE] ReID manifest written to /content/cctv-multiview-summarization/data_manifests/reid_summary.csv

Verifying ReID Outputs...
[PASS] view1_embeddings.npz exists with size 8699553 bytes.
[PASS] view2_embeddings.npz exists with size 10069398 bytes.
[PASS] view3_embeddings.npz exists with size 1

In [12]:
# ==============================================================================
# Step d: Run DINOv3 Feature Extraction
# ==============================================================================
print("\n" + "="*50)
print("[STEP D] Running DINOv3 Feature Extraction...")
print("="*50)
!python -m src.extract_dinov3_features --view all

print("\nVerifying DINOv3 Outputs...")
dino_files = [
    DRIVE_ROOT / "data" / "dinov3_embeddings" / "view1_dinov3.npz",
    DRIVE_ROOT / "data" / "dinov3_embeddings" / "view2_dinov3.npz",
    DRIVE_ROOT / "data" / "dinov3_embeddings" / "view3_dinov3.npz",
]

dino_passed = True
dino_results = {}
for f in dino_files:
    exists, size = verify_file(f)
    if exists:
        print(f"[PASS] {f.name} exists with size {size} bytes.")
        dino_results[f.name] = size
    else:
        print(f"[FAIL] {f.name} is missing or empty.")
        dino_passed = False

if not dino_passed:
    print("[FAIL] Step D failed verification. Halting.")
    sys.exit(1)


[STEP D] Running DINOv3 Feature Extraction...
Novelty #1: Full-Frame DINO Feature Extraction
Model ID  : facebook/dinov2-small
Target    : all
Batch Size: 32
----------------------------------------------------------------------
[INFO] Loading DINO model: facebook/dinov2-small onto cuda...
Loading weights: 100% 223/223 [00:00<00:00, 4833.37it/s]
[INFO] view1 DINO embeddings already exist at /content/drive/MyDrive/CCTV-Multiview-Project/data/dinov3_embeddings/view1_dinov3.npz. Skipping.
[INFO] view2 DINO embeddings already exist at /content/drive/MyDrive/CCTV-Multiview-Project/data/dinov3_embeddings/view2_dinov3.npz. Skipping.
[INFO] view3 DINO embeddings already exist at /content/drive/MyDrive/CCTV-Multiview-Project/data/dinov3_embeddings/view3_dinov3.npz. Skipping.
[SUCCESS] All targeted DINO feature extractions complete.

Verifying DINOv3 Outputs...
[PASS] view1_dinov3.npz exists with size 5567356 bytes.
[PASS] view2_dinov3.npz exists with size 5574575 bytes.
[PASS] view3_dinov3.npz

In [8]:
!python -m src.detect_objects --track --force

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[INFO] Loaded cached weights from /content/drive/MyDrive/CCTV-Multiview-Project/checkpoints/yolo26m.pt
[INFO] Using device: cuda (half_precision=True)
[INFO] Caching view1 frames to local disk (/tmp/frame_cache/view1)
[INFO] view1: 3912 frames cached locally
[INFO] Running detection on view1 (3912 frames, batch_size=32, track=True, motion_gating=False)
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantiz

In [13]:
# ==============================================================================
# Step e: Run Tracklet Feature Pooling
# ==============================================================================
print("\n" + "="*50)
print("[STEP E] Running Tracklet Feature Pooling...")
print("="*50)
!python -m src.pool_tracklet_features

print("\nVerifying Tracklet Pooling Output...")
tracklet_file = DRIVE_ROOT / "data" / "tracklet_embeddings.npz"
tracklet_passed = True
tracklet_results = {}
exists, size = verify_file(tracklet_file)
if exists:
    print(f"[PASS] {tracklet_file.name} exists with size {size} bytes.")
    tracklet_results[tracklet_file.name] = size
else:
    print(f"[FAIL] {tracklet_file.name} is missing or empty.")
    tracklet_passed = False

if not tracklet_passed:
    print("[FAIL] Step E failed verification. Halting.")
    sys.exit(1)


[STEP E] Running Tracklet Feature Pooling...
[SKIP] Outputs already present at /content/cctv-multiview-summarization/data_manifests/tracklet_features.csv and /content/drive/MyDrive/CCTV-Multiview-Project/data/tracklet_embeddings.npz (use --force to re-run)

Verifying Tracklet Pooling Output...
[PASS] tracklet_embeddings.npz exists with size 1140118 bytes.


In [14]:
# ==============================================================================
# Step f: Final Consolidated Report
# ==============================================================================
print("\n" + "="*50)
print("FINAL CONSOLIDATED REPORT")
print("="*50)
print("[SUCCESS] All 3 embedding types generated and verified on Drive.")
for k, v in reid_results.items():
    print(f"- {k}: {v} bytes \t(Drive: data/embeddings/{k})")
for k, v in dino_results.items():
    print(f"- {k}: {v} bytes \t(Drive: data/dinov3_embeddings/{k})")
for k, v in tracklet_results.items():
    print(f"- {k}: {v} bytes \t(Drive: data/{k})")
print("="*50)


FINAL CONSOLIDATED REPORT
[SUCCESS] All 3 embedding types generated and verified on Drive.
- view1_embeddings.npz: 8699553 bytes 	(Drive: data/embeddings/view1_embeddings.npz)
- view2_embeddings.npz: 10069398 bytes 	(Drive: data/embeddings/view2_embeddings.npz)
- view3_embeddings.npz: 11453451 bytes 	(Drive: data/embeddings/view3_embeddings.npz)
- view1_dinov3.npz: 5567356 bytes 	(Drive: data/dinov3_embeddings/view1_dinov3.npz)
- view2_dinov3.npz: 5574575 bytes 	(Drive: data/dinov3_embeddings/view2_dinov3.npz)
- view3_dinov3.npz: 5578455 bytes 	(Drive: data/dinov3_embeddings/view3_dinov3.npz)
- tracklet_embeddings.npz: 1140118 bytes 	(Drive: data/tracklet_embeddings.npz)
